# Internal IT Ticketing: Medallion Pipeline (PySpark / Microsoft Fabric)

A compilation of the Fabric PySpark notebooks I built to feed a dashboard for our internal IT ticketing system (ServiceNow incident, SLA and user data). Data flows through three Delta layers:

| Layer | Notebook section | Output tables |
|---|---|---|
| **Bronze** | raw ServiceNow extracts (input) | `inc_incident_bronze`, `inc_sys_user_bronze`, `inc_sla_bronze` |
| **Silver** | Part 1: Bronze → Silver | `inc_incident_silver`, `inc_sys_user_silver`, `inc_sla_silver` |
| **Gold** | Part 2: Silver → Gold | `inc_incident_gold`, `inc_sys_user_gold`, `inc_sla_gold`, `inc_agent_gold`, `dimdate` |

Sample CSVs of every layer are in the repo's `data/` folder so you can see the before/after of each step.


# Part 1: Bronze → Silver
Timezone conversion (UTC → US/Eastern), dropping unused columns, label mapping, and derived time metrics.


## Imports

In [ ]:
import pyspark.sql.functions as sf
from pyspark.sql import types as T
from pyspark.sql.functions import col, datediff, when, concat, lit, round as spark_round

## Creating Dataframes

In [ ]:
DB = "internal_it_100b_lkh"

incident_frame = spark.sql(f"SELECT * FROM {DB}.inc_incident_bronze")
sys_user_frame = spark.sql(f"SELECT * FROM {DB}.inc_sys_user_bronze")
sla_frame      = spark.sql(f"SELECT * FROM {DB}.inc_sla_bronze")

## Incident Frame

In [ ]:
def clean_data_incident(incident_frame):
    # Convert timestamps from UTC to Eastern
    incident_frame = incident_frame.withColumn("closed_at",      sf.from_utc_timestamp("closed_at",      "America/New_York")
    ).withColumn("opened_at",       sf.from_utc_timestamp("opened_at",       "America/New_York")
    ).withColumn("sys_updated_on",  sf.from_utc_timestamp("sys_updated_on",  "America/New_York"))

    # Drop irrelevant / null-heavy columns
    incident_frame = incident_frame.drop(
        'needs_attention', 'approval', 'knowledge', 'correlation_display', 'service_offering',
        'company', 'approval_set', 'contact_type', 'made_sla', 'expected_start', 'parent',
        'reassignment_count', 'due_date', 'order', 'group_list', 'activity_due',
        'business_service', 'skills', 'work_notes_list', 'reopen_count', 'sys_domain_path',
        'work_start', 'sys_class_name', 'upon_approval', 'hold_reason', 'parent_incident',
        'cause', 'delivery_task', 'work_end', 'comments', 'watch_list', 'sla_due',
        'escalation', 'time_worked', 'additional_assignee_list', 'correlation_id', 'contract',
        'cmdb_ci', 'approval_history', 'location', 'user_input', 'delivery_plan', 'work_notes',
        'follow_up', 'sys_domain', 'problem_id', 'universal_request', 'comments_and_work_notes',
        'child_incidents', 'business_impact', 'upon_reject', 'rejection_goto', 'caused_by',
        'notify', 'category', 'severity', 'reopened_by', 'origin_id', 'action_status', 'rfc',
        'subcategory', 'reopened_time', 'origin_table', 'variables', 'route_reason', 'wf_activity'
    )

    # Priority label
    incident_frame = incident_frame.withColumn('priority_text',
        when(col('priority') == 1, 'Critical')
       .when(col('priority') == 2, 'High')
       .when(col('priority') == 3, 'Medium')
       .when(col('priority') == 4, 'Low')
       .otherwise(None)
    )

    # State label
    incident_frame = incident_frame.withColumn("state_text",
        sf.when(incident_frame["state"] == 1, "New")
         .when(incident_frame["state"] == 2, "In Progress")
         .when(incident_frame["state"] == 3, "On Hold")
         .when(incident_frame["state"] == 6, "Resolved")
         .when(incident_frame["state"] == 7, "Closed")
         .otherwise("ERROR")
    )

    # State status rollup
    incident_frame = incident_frame.withColumn("state_status",
        sf.when(incident_frame["state_text"] == "New",         "Open")
         .when(incident_frame["state_text"] == "In Progress",  "Open")
         .when(incident_frame["state_text"] == "On Hold",      "Open")
         .when(incident_frame["state_text"] == "Resolved",     "Resolved")
         .when(incident_frame["state_text"] == "Closed",       "Closed")
         .otherwise("ERROR")
    )

    # Business resolve time in minutes, hours, and days
    incident_frame = incident_frame.withColumn('business_resolve_minutes', spark_round(col('business_stc') / 60,    4)
    ).withColumn('business_resolve_hours',   spark_round(col('business_stc') / 3600,  4)
    ).withColumn('business_resolve_days',    spark_round(col('business_stc') / 86400, 4))

    # Cast active to string
    incident_frame = incident_frame.withColumn('active', incident_frame['active'].cast(T.StringType()))

    # Day of week number (0=Monday, 6=Sunday — matches Python dt.weekday)
    incident_frame = incident_frame.withColumn("created_day_of_week#",
        sf.when(sf.dayofweek(incident_frame["opened_at"]) == 1, 6)
         .otherwise(sf.dayofweek(incident_frame["opened_at"]) - 2)
    )

    # Day of week label
    incident_frame = incident_frame.withColumn("created_day",
        sf.when(incident_frame["created_day_of_week#"] == 0, "Monday")
         .when(incident_frame["created_day_of_week#"] == 1, "Tuesday")
         .when(incident_frame["created_day_of_week#"] == 2, "Wednesday")
         .when(incident_frame["created_day_of_week#"] == 3, "Thursday")
         .when(incident_frame["created_day_of_week#"] == 4, "Friday")
         .otherwise("Weekend")
    )

    # Hour number and label
    incident_frame = incident_frame.withColumn("created_hour#", sf.hour(incident_frame["opened_at"]))
    incident_frame = incident_frame.withColumn('created_hour',
        sf.when(col('created_hour#') < 12,  concat(col('created_hour#').cast('string'),          lit(' AM')))
         .when(col('created_hour#') > 12,   concat((col('created_hour#') - 12).cast('string'),   lit(' PM')))
         .when(col('created_hour#') == 12,  concat(col('created_hour#').cast('string'),           lit(' PM')))
    )

    # Collapse early morning and late evening into "After Hours"
    incident_frame = incident_frame.withColumn('created_hour',
        sf.when(
            ((col('created_hour#') >= 0)  & (col('created_hour#') <= 8)) |
            ((col('created_hour#') >= 18) & (col('created_hour#') <= 24)),
            'After Hours'
        ).otherwise(col('created_hour'))
    )

    return incident_frame

# Apply cleaning functions
incident_frame = clean_data_incident(incident_frame)

## User Frame

In [ ]:
def clean_data_sys_user(sys_user_frame):
    # Drop irrelevant columns
    sys_user_frame = sys_user_frame.drop(
        'country', 'mobile_phone', 'time_format', 'calendar_integration', 'street',
        'default_perspective', 'state', 'cost_center', 'roles', 'last_password',
        'employee_number', 'phone', 'time_zone', 'building', 'last_position_update',
        'enable_multifactor_authn', 'time_sheet_policy', 'u_customer_account', 'ldap_server',
        'hashed_user_id', 'user_password', 'accumulated_roles', 'last_login_device',
        'edu_status', 'vip', 'sso_source', 'latitude', 'preferred_language', 'home_phone',
        'date_format', 'notification', 'source', 'failed_attempts', 'introduction', 'photo',
        'city', 'sys_domain', 'middle_name', 'gender', 'locked_out', 'location', 'zip',
        'avatar', 'internal_integration_user', 'sys_domain_path', 'identity_type',
        'geolocation_tracked', 'schedule', 'fax', 'agent_status', 'on_schedule',
        'longitude', 'company'
    )

    # Rename ambiguous column
    sys_user_frame = sys_user_frame.withColumnRenamed('name', 'name_id')

    return sys_user_frame

# Apply cleaning functions
sys_user_frame = clean_data_sys_user(sys_user_frame)

## Sla Frame

In [ ]:
def clean_data_sla(sla_frame):
    # Business duration baseline and derived columns
    sla_frame = sla_frame.withColumn('duration_start', sf.lit('1970-01-01 00:00:00').cast('timestamp'))
    sla_frame = sla_frame.withColumn('business_duration_day#',  sf.datediff(sla_frame.taskslatable_business_duration, sla_frame.duration_start))
    sla_frame = sla_frame.withColumn('business_duration_secs',  sf.col("taskslatable_business_duration").cast("long") - sf.col("duration_start").cast("long"))
    sla_frame = sla_frame.withColumn('business_duration_minutes', spark_round(col('business_duration_secs') / 60,    4)
    ).withColumn('business_duration_hours',  spark_round(col('business_duration_secs') / 3600,  4)
    ).withColumn('business_duration_days',   spark_round(col('business_duration_secs') / 86400, 4))

    # Convert timestamps from UTC to Eastern
    sla_frame = (sla_frame
        .withColumn("taskslatable_original_breach_time", sf.from_utc_timestamp("taskslatable_original_breach_time", "America/New_York"))
        .withColumn("taskslatable_end_time",              sf.from_utc_timestamp("taskslatable_end_time",              "America/New_York"))
        .withColumn("taskslatable_start_time",            sf.from_utc_timestamp("taskslatable_start_time",            "America/New_York"))
        .withColumn("taskslatable_sys_created_on",        sf.from_utc_timestamp("taskslatable_sys_created_on",        "America/New_York"))
        .withColumn("taskslatable_planned_end_time",      sf.from_utc_timestamp("taskslatable_planned_end_time",      "America/New_York"))
        .withColumn("taskslatable_sys_updated_on",        sf.from_utc_timestamp("taskslatable_sys_updated_on",        "America/New_York"))
        .withColumn("inc_sys_updated_on",                 sf.from_utc_timestamp("inc_sys_updated_on",                 "America/New_York"))
        .withColumn("inc_sys_created_on",                 sf.from_utc_timestamp("inc_sys_created_on",                 "America/New_York"))
        .withColumn("inc_closed_at",                      sf.from_utc_timestamp("inc_closed_at",                      "America/New_York"))
        .withColumn("inc_opened_at",                      sf.from_utc_timestamp("inc_opened_at",                      "America/New_York"))
        .withColumn("inc_resolved_at",                    sf.from_utc_timestamp("inc_resolved_at",                    "America/New_York"))
    )

    # Drop irrelevant SLA columns
    sla_frame = sla_frame.drop(
        'taskslatable_timezone', 'taskslatable_pause_time', 'taskslatable_schedule',
        'inc_approval', 'inc_knowledge', 'inc_correlation_display', 'inc_service_offering',
        'inc_company', 'inc_approval_set', 'inc_expected_start', 'inc_parent', 'inc_due_date',
        'inc_order', 'inc_group_list', 'inc_activity_due', 'inc_business_service', 'inc_skills',
        'inc_work_notes_list', 'inc_reopen_count', 'inc_sys_domain_path', 'inc_work_start',
        'inc_sys_class_name', 'inc_upon_approval', 'inc_hold_reason', 'inc_parent_incident',
        'inc_cause', 'inc_delivery_task', 'inc_work_end', 'inc_comments', 'inc_watch_list',
        'inc_sla_due', 'inc_escalation', 'inc_time_worked', 'inc_additional_assignee_list',
        'inc_correlation_id', 'inc_contract', 'inc_cmdb_ci', 'inc_approval_history',
        'inc_location', 'inc_user_input', 'inc_delivery_plan', 'inc_work_notes', 'inc_follow_up',
        'inc_sys_domain', 'inc_problem_id', 'inc_universal_request', 'inc_comments_and_work_notes',
        'inc_child_incidents', 'inc_business_impact', 'inc_upon_reject', 'inc_rejection_goto',
        'inc_caused_by', 'inc_notify', 'inc_category', 'inc_severity', 'inc_reopened_by',
        'inc_origin_id', 'inc_action_status', 'inc_rfc', 'inc_subcategory', 'inc_reopened_time',
        'inc_origin_table', 'inc_variables', 'inc_route_reason', 'inc_wf_activity'
    )
    sla_frame = sla_frame.drop(
        'inc_resolved_at', 'inc_calendar_stc', 'inc_incident_state', 'inc_business_stc',
        'inc_caller_id', 'inc_task_effective_number', 'inc_closed_by',
        'inc_state', 'inc_active', 'inc_sys_updated_by', 'inc_assigned_to', 'inc_business_duration',
        'inc_urgency', 'inc_opened_at', 'inc_sys_mod_count', 'inc_close_code', 'inc_resolved_by',
        'inc_closed_at', 'inc_impact', 'inc_description', 'inc_sys_created_on', 'inc_close_notes',
        'inc_reassignment_count', 'inc_priority', 'inc_sys_updated_on', 'inc_made_sla',
        'inc_contact_type', 'inc_opened_by', 'inc_short_description', 'inc_calendar_duration',
        'inc_assignment_group', 'inc_needs_attention'
    )

    return sla_frame

# Apply cleaning functions
sla_frame      = clean_data_sla(sla_frame)

## Saving Tables

In [ ]:
SILVER_DB = "internal_it_200s_lkh"

silver_frames = {
    "inc_incident_silver": incident_frame,
    "inc_sys_user_silver": sys_user_frame,
    "inc_sla_silver":      sla_frame,
}

for table_name, df in silver_frames.items():
    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{SILVER_DB}.{table_name}")

# Part 2: Silver → Gold
Business-ready tables: filtered incidents/users, name lookups for agents, a US-workday date dimension, and dashboard helper columns.


## Imports

In [ ]:
# Standard library
from datetime import date

# Third-party
import pandas as pd
import holidays     # https://pypi.org/project/holidays/
from pandas.tseries.offsets import MonthEnd, QuarterEnd

# PySpark
import pyspark.sql.functions as sf
from pyspark.sql import types as T
from pyspark.sql import Window, Row
from pyspark.sql.functions import (
    col,
    concat,
    datediff,
    expr,
    lit,
    monotonically_increasing_id,
    row_number,
    substring,
    when,
    round as spark_round,
)

## Date Dimension

In [ ]:
end_date = date.today().strftime("%Y-%m-%d")

def DimDate(start_date, end_date):
    df_date = pd.DataFrame({"Date": pd.date_range(start=f'{start_date}', end=f'{end_date}', freq='D')})

    def get_end_of_month(pd_date):
        if pd_date.is_month_end == True:
            return pd_date
        else:
            return pd_date + MonthEnd(1)

    def get_end_of_quarter(pd_date):
        if pd_date.is_quarter_end == True:
            return pd_date
        else:
            return pd_date + QuarterEnd(1)

    df_date["Day"] = df_date.Date.dt.day
    df_date["Week"] = df_date.Date.dt.weekday
    df_date["Month"] = df_date.Date.dt.month
    df_date["MonthName"] = df_date.Date.dt.month_name()
    df_date["Quarter"] = df_date.Date.dt.quarter
    df_date["Year"] = df_date.Date.dt.year
    df_date['EndOfMonth'] = df_date['Date'].apply(get_end_of_month)
    df_date['EOM'] = df_date['Date'].dt.is_month_end
    df_date['EndOfQuarter'] = df_date['Date'].apply(get_end_of_quarter)
    df_date['EOQ'] = df_date['Date'].dt.is_quarter_end
    return df_date

def DimDateWorkday(start_date, end_date):
    df_date = DimDate(start_date, end_date)
    df_date['Workday'] = True
    for index, row in df_date.iterrows():
        if row['Week'] in [5, 6]:  # Saturday=5, Sunday=6
            df_date.loc[index, 'Workday'] = False
        date = row['Date'].strftime("%Y-%m-%d")
        if date in holidays.US():
            df_date.loc[index, 'Workday'] = False
    return df_date

DimDate_frame = spark.createDataFrame(DimDateWorkday('2025-3-1',end_date))

DimDate_frame = (DimDate_frame
    .withColumn('MonthYear',   concat(col('MonthName'), lit(' '), col('Year').cast('string')))
    .withColumn('QuarterYear', concat(col('Year'), lit(' Q'), col('Quarter').cast('string')))
    .orderBy("Date")
)

## Load Silver Incident Data

In [ ]:
DB = "internal_it_200s_lkh"

#Pulls necessary columns
incident_frame  = spark.sql(f"SELECT * FROM {DB}.inc_incident_silver")
sys_user_frame  = spark.sql(f"SELECT * FROM {DB}.inc_sys_user_silver")
sla_frame       = spark.sql(f"SELECT * FROM {DB}.inc_sla_silver")


## Incident Frame

In [ ]:
def clean_data_incident(incident_frame):
    #Remove values for resolve time that is 0 minutes (instant)
    incident_frame = incident_frame.withColumn('business_resolve_minutes', when(col('business_resolve_minutes') == 0, None).otherwise(col('business_resolve_minutes'))).withColumn('business_resolve_hours', when(col('business_resolve_hours') == 0, None).otherwise(col('business_resolve_hours')))
    #Add columns
    incident_frame = incident_frame.withColumn('sys_created_date', col('sys_created_on'))
    incident_frame = incident_frame.withColumn('sys_created_date', substring('sys_created_date', 1, 10))
    incident_frame = incident_frame.withColumn('active_state', sf.when(incident_frame["active"] == "true", 1).otherwise(0))
    #Creating a sort column so after hours appears at the end of Tickets By Hour
    incident_frame = incident_frame.withColumn('created_hour_sort', col('created_hour#'))
    incident_frame = incident_frame.withColumn('created_hour_sort', when((col('created_hour_sort') >= 9) & (col('created_hour_sort') <= 17), col('created_hour_sort')).otherwise(25))
    #Removing tickets that are category Security that were opened by service-account user
    incident_frame = incident_frame.filter(
        ~(
            (col('opened_by') == '7333494e2bb14750daddf57ace91bf91')  &
            #(col('assigned_to') == '7333494e2bb14750daddf57ace91bf91') &
            (col('assignment_group') == '3f176dd83b816610cfa36d54c3e45ae3')
        )
    )
    
    

    return incident_frame
incident_frame = clean_data_incident(incident_frame)

## SLA Frame

In [ ]:
def clean_data_sla(sla_frame):
    return sla_frame
sla_frame = clean_data_sla(sla_frame)

## Resolved By, Assigned To, Closed By, and Opened By Table

This is to create a table that translates user IDs to names for the corresponding ticket (incident number)

In [ ]:
DB = "internal_it_200s_lkh"

incident_name_frame = spark.sql(f"SELECT number, opened_by, assigned_to, resolved_by, closed_by FROM {DB}.inc_incident_silver")
user_frame = spark.sql(f"SELECT sys_id, name_id FROM {DB}.inc_sys_user_silver")

# Alias sys_user_frame once per role to avoid column ambiguity across joins
su_opened   = user_frame.alias("su_opened")
su_assigned = user_frame.alias("su_assigned")
su_resolved = user_frame.alias("su_resolved")
su_closed   = user_frame.alias("su_closed")

# Single chained join — each alias maps unambiguously to its field
agent_name_frame = (
    incident_name_frame
    .join(su_opened,   col("opened_by")   == col("su_opened.sys_id"),   "left")
    .join(su_assigned, col("assigned_to") == col("su_assigned.sys_id"), "left")
    .join(su_resolved, col("resolved_by") == col("su_resolved.sys_id"), "left")
    .join(su_closed,   col("closed_by")   == col("su_closed.sys_id"),   "left")
    .select(
        col("number"),
        col("su_opened.name_id").alias("opened_by_name"),
        col("su_assigned.name_id").alias("assigned_to_name"),
        col("su_resolved.name_id").alias("resolved_by_name"),
        col("su_closed.name_id").alias("closed_by_name"),   # ← now guaranteed distinct from resolved_by_name
    )
    .sort(sf.desc("number"))
    .fillna("Unassigned", subset=["assigned_to_name"])
)

## User Frame

In [ ]:
def clean_data_sys_user(sys_user_frame):
    #Removes non-employees
    sys_user_frame = sys_user_frame.filter(col('email').endswith('@company.com'))
    return sys_user_frame
sys_user_frame = clean_data_sys_user(sys_user_frame)
display(sys_user_frame)

## Saving Tables

In [ ]:
#Saving Tables
gold_frames = {
    "inc_incident_gold": incident_frame,
    "inc_sys_user_gold": sys_user_frame,
    "inc_sla_gold":  sla_frame,
    "inc_agent_gold":  agent_name_frame,
    "dimdate": DimDate_frame,
}

GOLD_DB = "internal_it_300g_lkh"

for table_name, df in gold_frames.items():
    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{GOLD_DB}.{table_name}")